In [ ]:
# Cell 1: Install Dependencies
%%capture
import torch, re
v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
xformers = 'xformers==' + {'2.10': '0.0.34', '2.9': '0.0.33.post1', '2.8': '0.0.32.post2'}.get(v, "0.0.34")
!pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
!pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
!pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

In [ ]:
# Cell 2: Load the base model and attach a LoRA adapter

from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None             # auto-detect: float16 on a T4, bfloat16 on newer GPUs
load_in_4bit = True      # quantization #1: load the base model in 4-bit

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Llama-3.2-3B-Instruct-bnb-4bit",  # same base model as L2's `ollama run llama3.2`
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,                 # LoRA rank - size of the adapter
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                     "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    use_rslora=False,
    loftq_config=None,
)

In [ ]:
# Cell 3: Upload and load the training data

from google.colab import files
uploaded = files.upload()  # select tagged_transactions.jsonl from your computer

from datasets import load_dataset
dataset = load_dataset("json", data_files="tagged_transactions.jsonl", split="train")

In [ ]:
# Cell 4: Format each row into the model's chat template

from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template="llama-3.1",  # Llama 3.2's chat template is the same as 3.1's
)

SYSTEM_PROMPT = (
    "You are a bank transaction categorizer. Given a transaction's description "
    "and amount, respond with only a JSON object with these fields: merchant, "
    "category (one of: dining, groceries, fuel, transport, subscriptions, "
    "utilities, entertainment, health, retail, cash_withdrawal, income, "
    "transfers), channel (one of: card, direct_debit, bank_transfer, atm), and "
    "flagged_for_review (true or false)."
)

def formatting_prompts_func(examples):
    texts = []
    for input_str, output_str in zip(examples["input"], examples["output"]):
        convo = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": input_str},
            {"role": "assistant", "content": output_str},
        ]
        texts.append(tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False))
    return {"text": texts}

dataset = dataset.map(formatting_prompts_func, batched=True)

In [ ]:
# Cell 5: Configure the trainer

from trl import SFTConfig, SFTTrainer
from transformers import DataCollatorForSeq2Seq

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer),
    packing=False,
    args=SFTConfig(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        max_steps=200,
        learning_rate=2e-4,
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.001,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="outputs",
        report_to="none",
    ),
)

from unsloth.chat_templates import train_on_responses_only
trainer = train_on_responses_only(trainer)  # only compute loss on the assistant's JSON, not the prompt

In [ ]:
# Cell 6: Run training

trainer_stats = trainer.train()

In [ ]:
# Cell 7: Test model on an unseen input

import json

FastLanguageModel.for_inference(model)  # switch from training mode to fast inference

example = {"description": "SQ *BLUE BOTTLE COFFEE 4521 PORTLAND OR", "amount": -7.20}

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": json.dumps(example)},
]
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
).to("cuda")

outputs = model.generate(input_ids=inputs, max_new_tokens=128, use_cache=True, do_sample=False)
response = tokenizer.batch_decode(outputs[:, inputs.shape[1]:], skip_special_tokens=True)[0]
print(f"{example['description']} -> {response}")

In [ ]:
# Cell 8: Save the adapter and export to GGUF

model.save_pretrained("lora_model")
tokenizer.save_pretrained("lora_model")

# Merges the adapter into the base, model and writes it out as a quantized GGUF file, ready for Ollama
model.save_pretrained_gguf("tx_categorizer", tokenizer, quantization_method="q4_k_m")

In [ ]:
# Cell 9: Download the GGUF and Modelfile locally

from google.colab import files
files.download("tx_categorizer_gguf/Llama-3.2-3B-Instruct.Q4_K_M.gguf")
files.download("tx_categorizer_gguf/Modelfile")